# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

In [ ]:
# load necessary packages
# copied from lecture 9 

import re
import Bio.SeqIO
import Bio.Seq

In [ ]:
# get sequences
# copied from lecture 9

seqreads = list(Bio.SeqIO.parse('barcodes_R1.fastq', 'fastq'))

seqreads_Seq = []
for seqrecord in seqreads:
    seqreads_Seq.append(seqrecord.seq)

In [ ]:
## copied this over from lecture 9
## this code will read and return the barcode of a sequence

def read_barcode(seqread, bclen = 16, upstream='AGGCGGCCGC'):
    """Identify barcode with known upstream sequence.
    
    Parameters
    ----------
    seqread : Seq object
        Nucleotide sequence read matching UPSTREAM-BARCODE in reverse orientation.
    bclen : int
        Length of barcode
    upstream: str
        Sequence upstream of the barcode.
        
    Returns
    -------
    str or None
        Sequence of the barcode in the forward orientation, or `None` if no match to expected barcoded sequence.
        
    Example
    -------
    >>> read_barcode(Bio.Seq.Seq('TTTTTTTTTTTTTTTTGCGGCCGCCT'), bclen=16)
    'AAAAAAAAAAAAAAAA'
        
    """
    
    seq_not_str = seqread.reverse_complement()
    seq_str = str(seq_not_str) 

    barcode_re = re.compile(upstream + "(?P<barcode>[ATCG]{" + str(bclen) + "})$")
    
    match = barcode_re.search(seq_str)
    if match:
        barcode = match.group("barcode")
    else:
        barcode = None
    
    return barcode


In [ ]:
# Initializes two variables that are the end sequences of NA and HA
NA_END = 'CACGATAGATAAATAATAGTGCACCAT'
HA_END = 'CCGGATTTGCATATAATGATGCACCAT'


def get_strain_type_barcodes(seqread, NA_dict, HA_dict, downstream='AGGCGGCCGC'):
    """Identifies whether the sequence comes from the HA or NA strain
        and then adds the barcode associated with that sequence to the barcode dictionaries
        initialized above.
    
    Parameters
    ----------
    seqread : Seq object
        Nucleotide sequence read matching UPSTREAM-BARCODE in reverse orientation.
    NA_dict: dict object
        takes in a dictonary and appends it with the barcode if the given sequence is an NA strain
    HA_dict: dict object
        takes in a dictonary and appends it with the barcode if the given sequence is an HA strain
    downstream: str
        Sequence downstream of the HA/NA sequence.
        
    Returns
    -------
    returns the updated HA/NA barcode dictionaries.
        
    """

    # creates search
    strain_type = re.compile("(?P<barcode>[ATCG]{27})" + downstream)

    # converts reverse complement to a string for searching
    seq_not_str = seqread.reverse_complement()
    seq_str = str(seq_not_str) 
    

    strain_temp = strain_type.search(seq_str)

    if strain_temp:
        strain = strain_temp.group('barcode')
        if strain == NA_END:

            seq_barcode = read_barcode(seqread, bclen = 16)
            if seq_barcode:
                if len(seq_barcode) in NA_dict:
                    NA_dict[seq_barcode] += 1
                else:
                    NA_dict[seq_barcode] = 1
            else:
                NA_dict['invalid'] += 1

        elif strain == HA_END:

            seq_barcode = read_barcode(seqread, bclen = 16)
            if seq_barcode:
                if len(seq_barcode) in HA_dict:
                    HA_dict[seq_barcode] += 1
                else:
                    HA_dict[seq_barcode] = 1
            else:
                HA_dict['invalid'] += 1

        
    return NA_dict, HA_dict

# # Test Case to ensure function runs
assert isinstance(get_strain_type_barcodes(seqreads_Seq[0], {}, {}), tuple)

In [39]:
# Initializes the dictionaries we will use to count barcodes for each strain
BARCODE_COUNTS_NA = {'invalid':0}
BARCODE_COUNTS_HA = {'invalid':0}

for seq in seqreads_Seq:
    get_strain_type_barcodes(seq, BARCODE_COUNTS_NA, BARCODE_COUNTS_HA)

print(f"number of NA barcodes: {len(BARCODE_COUNTS_NA)}")
print(f"number of HA barcodes: {len(BARCODE_COUNTS_HA)}")

number of NA barcodes: 136
number of HA barcodes: 281


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [41]:
# Prints the number of invalid barcodes for each strain

print(f"number of invalid NA barcodes: {BARCODE_COUNTS_NA['invalid']}")
print(f"number of invlaid HA barcodes: {BARCODE_COUNTS_HA['invalid']}")

number of invalid NA barcodes: 66
number of invlaid HA barcodes: 71


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [42]:
# finds that max value's key in the given dictionary
max_NA = max(BARCODE_COUNTS_NA, key = BARCODE_COUNTS_NA.get)
max_HA = max(BARCODE_COUNTS_HA, key = BARCODE_COUNTS_HA.get)

# prints the max count key and counts
print(f'NA barcode with most counts is \'{max_NA}\' with {BARCODE_COUNTS_NA[max_NA]} counts')
print(f'HA barcode with most counts is: \'{max_HA}\' with {BARCODE_COUNTS_HA[max_HA]} counts')



NA barcode with most counts is 'invalid' with 66 counts
HA barcode with most counts is: 'invalid' with 71 counts
